In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
import asyncio
import time
from decimal import Decimal

In [2]:
from decimal import Decimal, getcontext
import math

def ensure_multiple_of_step_size(margin_quantity, step_size_str):
    """
    Ensures that margin_quantity is a multiple of step_size.
    If not, it truncates margin_quantity *down* to the nearest multiple.

    Args:
        margin_quantity: The number to adjust (can be a float or Decimal).
        step_size_str: The step size as a string (e.g., "0.00100000").

    Returns:
        The adjusted margin_quantity as a Decimal.  (Using Decimal avoids float precision issues)
    """

    step_size = Decimal(step_size_str) # Convert step_size to Decimal immediately
    
    #If margin_quantity is a float, convert to decimal for accuracy
    if isinstance(margin_quantity, float):
      margin_quantity = Decimal(str(margin_quantity))

    #Calculate the remainder when margin_quantity is divided by step_size
    remainder = margin_quantity % step_size

    # if remainder != 0:
    #   #If there is a remainder, subtract the remainder from margin_quantity
    #   margin_quantity -= remainder
    
    return margin_quantity


# # Example usage
# step_size = '0.00100000'

# # Test cases (float or Decimal)
# margin_quantity1 = 0.0053
# margin_quantity2 = Decimal('0.0107')
# margin_quantity3 = 0.123456789
# margin_quantity4 = Decimal('10.99999')
# margin_quantity5 = 0.001  # Already a multiple

# adjusted_quantity1 = ensure_multiple_of_step_size(margin_quantity1, step_size)
# adjusted_quantity2 = ensure_multiple_of_step_size(margin_quantity2, step_size)
# adjusted_quantity3 = ensure_multiple_of_step_size(margin_quantity3, step_size)
# adjusted_quantity4 = ensure_multiple_of_step_size(margin_quantity4, step_size)
# adjusted_quantity5 = ensure_multiple_of_step_size(margin_quantity5, step_size)


# print(f"Original: {margin_quantity1}, Adjusted: {adjusted_quantity1}")  # Original: 0.0053, Adjusted: 0.005
# print(f"Original: {margin_quantity2}, Adjusted: {adjusted_quantity2}")  # Original: 0.0107, Adjusted: 0.010
# print(f"Original: {margin_quantity3}, Adjusted: {adjusted_quantity3}")  # Original: 0.123456789, Adjusted: 0.123
# print(f"Original: {margin_quantity4}, Adjusted: {adjusted_quantity4}")  # Original: 10.99999, Adjusted: 10.999
# print(f"Original: {margin_quantity5}, Adjusted: {adjusted_quantity5}")  # Original: 0.001, Adjusted: 0.001

In [14]:
base_asset = 'ETH'
quote_asset = 'USD'
symbol = f'{base_asset}{quote_asset}_PERP' # e.g., "BTCUSD_PERP"
trade_type = 'long'  # or 'short'
order_price = 2687
take_profit_price = 2714
stop_loss_price = 2675
risk_amount = 20.0  # USDT to risk
margin_leverage = 5
side_multiplier = 1 if trade_type == 'long' else -1

calculate_position_size(order_price, take_profit_price, stop_loss_price, risk_amount, margin_leverage=5, trade_type="long")


(0.3333333333333333, 895.6666666666666, 12, 20.0, 45.0)

In [13]:

def calculate_position_size(current_price, target_price, stop_loss_price, risk_amount, margin_leverage=5, trade_type="long"):
    """
    Calculates the position size (units to buy/sell) based on fixed margin,
    risk amount, and stop loss.
    (Same function as in the previous response)
    """
    if not isinstance(current_price, (int, float)) or not isinstance(target_price, (int, float)) or \
       not isinstance(stop_loss_price, (int, float)) or not isinstance(risk_amount, (int, float)) or \
       not isinstance(margin_leverage, int) or not isinstance(trade_type, str):
        print("Error: Invalid input types. Prices and risk_amount should be numbers, margin_leverage should be an integer, trade_type should be a string.")
        return None

    if risk_amount <= 0:
        print("Error: Risk amount must be positive.")
        return None

    if margin_leverage <= 1:
        print("Error: Margin leverage should be greater than 1.")
        return None

    trade_type = trade_type.lower()
    if trade_type not in ["long", "short"]:
        print("Error: Invalid trade_type. Must be 'long' or 'short'.")
        return None

    if current_price <= 0 or target_price <= 0 or stop_loss_price <= 0:
        print("Error: Prices must be positive.")
        return None

    if trade_type == "long":
        if stop_loss_price >= current_price:
            print("Error: Stop loss price for a long position must be lower than the current price.")
            return None
        risk_per_unit = current_price - stop_loss_price
        profit_per_unit = target_price - current_price

    elif trade_type == "short":
        if stop_loss_price <= current_price:
            print("Error: Stop loss price for a short position must be higher than the current price.")
            return None
        risk_per_unit = stop_loss_price - current_price
        profit_per_unit = current_price - target_price

    if risk_per_unit == 0:
        print("Error: Risk per unit is zero. Stop loss and current price are the same. Cannot calculate position size.")
        return None

    units = risk_amount / (risk_per_unit * margin_leverage)
    position_value = units * current_price
    potential_loss = units * risk_per_unit * margin_leverage
    potential_profit = units * profit_per_unit * margin_leverage

    return units, position_value, risk_per_unit, potential_loss, potential_profit



## Margin Orders with OCO

In [ ]:

import logging

from binance.spot import Spot as Client
from binance.lib.utils import config_logging

api_key = os.getenv('BINANCE_API_KEY')
api_secret = os.getenv('BINANCE_SECRET_KEY')


client = Client(api_key, api_secret)
exchange_info = client.exchange_info()

all_pairs = client.margin_all_pairs()


In [ ]:


def place_margin_order_with_oco(
    symbol,
    take_profit_price,
    stop_loss_price,
    risk_amount,
    margin_leverage,
    trade_type='long',  # or 'short'
):
    """
    Places a margin order with a corresponding OCO order for take profit and stop loss.

    Args:
        take_profit_price (float): The price at which to take profit.
        stop_loss_price (float): The price at which to stop loss.
        risk_amount (float): The amount of USDT to risk.
        trade_type (str): 'long' or 'short' to indicate the direction of the trade.
    """
    global base_asset, quote_asset  # Access the global variables

    try:
        # Symbol and Step Size retrieval (Error handling improved)
        try:
            symbol = [pair for pair in all_pairs if pair['base'] == base_asset and pair['quote'] == quote_asset][0]['symbol']
        except IndexError:
            print(f"Error: No trading pair found for {base_asset}/{quote_asset}")
            return

        try:
            symbol_info = [s for s in exchange_info['symbols'] if s['symbol'] == symbol][0]
        except IndexError:
            print(f"Error: Symbol information not found for symbol {symbol} in exchange_info")
            return

        try:
            step_size = ([f for f in symbol_info['filters'] if f['filterType'] == 'LOT_SIZE'][0]['stepSize'])
        except IndexError:
            print(f"Error: LOT_SIZE filter not found for symbol {symbol}")
            return

        # Get the current price
        try:
            ticker = client.ticker_price(symbol)
            current_price = float(ticker['price'])
        except Exception as e:
            print(f"Error fetching ticker price: {e}")
            return

        # Calculate position data
        position_data = calculate_position_size(current_price, take_profit_price, stop_loss_price, risk_amount, margin_leverage, trade_type)
        units, position_value, risk_per_unit, potential_loss, potential_profit = position_data

        print(f"Calculated Units: {units:.6f}, Position Value: {position_value:.2f}, Risk per Unit: {risk_per_unit:.2f}, Potential Loss: {potential_loss:.2f}, Potential Profit: {potential_profit:.2f}")

        is_isolated = "FALSE"  # "TRUE" for isolated margin, "FALSE" for cross margin

        fee_percent = 0.001  # 0.001 for spot, 0.0005 for futures

        # First Margin Order Configuration:  Be VERY careful with amounts!
        margin_side = "BUY" if trade_type == "long" else "SELL"  # or "SELL" for short
        margin_type = "MARKET"  # "MARKET", "LIMIT", etc.
        margin_quantity = ensure_multiple_of_step_size(units, step_size)

        # OCO Order Configuration (Take Profit and Stop Loss)
        oco_side = "SELL" if margin_side == "BUY" else "BUY"  # Opposite of the margin order
        oco_quantity = ensure_multiple_of_step_size(units * (1 - fee_percent), step_size)

        # Create the initial margin order
        try:
            margin_order_response = client.new_margin_order(
                symbol=symbol,
                side=margin_side,
                type=margin_type,
                quantity=margin_quantity,
                isIsolated=is_isolated
                #price=margin_price if margin_type == "LIMIT" else None
            )

            if margin_order_response:
                print("Margin Order Response:", margin_order_response)

                # Create the OCO order immediately after
                oco_order_response = client.new_margin_oco_order(
                    symbol=symbol,
                    side=oco_side,
                    quantity=oco_quantity,
                    price=take_profit_price,
                    stopPrice=stop_loss_price,
                    isIsolated=is_isolated
                )

                if oco_order_response:
                    print("Margin OCO Order Response:", oco_order_response)
                else:
                    print("Failed to create Margin OCO order.")
            else:
                print("Failed to create initial Margin order.")

        except Exception as e:
            print(f"An error occurred during order placement: {e}")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    print("Script finished.")


base_asset = 'BTC'
quote_asset = 'USDT'
trade_type = 'short'  # or 'short'
take_profit_price = 96760
stop_loss_price = 98190
risk_amount = 10.0  # USDT to risk
margin_leverage = 10
# Example usage (assuming all_pairs, exchange_info, client are defined elsewhere)
place_margin_order_with_oco(
    take_profit_price,
    stop_loss_price,
    risk_amount,
    margin_leverage,
    trade_type=trade_type,
)

Calculated Units: 0.001878, Position Value: 183.42, Risk per Unit: 532.42, Potential Loss: 10.00, Potential Profit: 16.86
An error occurred during order placement: (400, -2010, 'Account has insufficient balance for requested action.', {'Content-Type': 'application/json', 'Content-Length': '77', 'Connection': 'keep-alive', 'Date': 'Thu, 13 Feb 2025 00:27:23 GMT', 'Server': 'nginx', 'X-SAPI-USED-UID-WEIGHT-1M': '6', 'x-mbx-order-count-10s': '1', 'x-mbx-uuid': 'fcebb5de-1869-4ae3-8cb7-daa612d620ea', 'x-mbx-order-count-1d': '1', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains', 'X-Frame-Options': 'SAMEORIGIN', 'X-Xss-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'Content-Security-Policy': "default-src 'self'", 'X-Content-Security-Policy': "default-src 'self'", 'X-WebKit-CSP': "default-src 'self'", 'Cache-Control': 'no-cache, no-store, must-revalidate', 'Pragma': 'no-cache', 'Expires': '0', 'X-Cache': 'Error from cloudfront', 'Via': '1.1 85b9b6c170ed4e

## Binance Futures with OCO

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
import asyncio
import time
from decimal import Decimal
from binance.cm_futures import CMFutures


api_key = os.getenv('BINANCE_API_KEY')
api_secret = os.getenv('BINANCE_SECRET_KEY')

client = CMFutures(key=api_key, secret=api_secret)
exchange_info = client.exchange_info()


# exchange_info = client.exchange_info()
all_pairs = exchange_info['symbols']


In [ ]:
base_asset = 'ETH'
quote_asset = 'USD'
symbol = f'{base_asset}{quote_asset}_PERP' # e.g., "BTCUSD_PERP"
trade_type = 'long'  # or 'short'
order_price = 2687
take_profit_price = 2714
stop_loss_price = 2675
risk_amount = 20.0  # USDT to risk
margin_leverage = 5
side_multiplier = 1 if trade_type == 'long' else -1
# symbol_info = [pair for pair in all_pairs if pair['symbol'] == symbol][0]

# symbol_info_filter = [f for f in symbol_info['filters'] if f['filterType'] == 'LOT_SIZE'][0]
# step_size = symbol_info_filter['stepSize']
# min_qty = float(symbol_info_filter['minQty'])
min_qty = 0.0002

# ticker = client.ticker_price(symbol)[0]
# current_price = float(ticker['price'])
# Current Price: {current_price:.4f}, 
print(f"Order Price: {order_price:.4f}, Take Profit: {take_profit_price:.4f}, Stop Loss: {stop_loss_price:.4f}")

risk_per_unit = (order_price - stop_loss_price) #* side_multiplier
profit_per_unit = (take_profit_price - order_price) #* side_multiplier
units = risk_amount / (risk_per_unit * margin_leverage) #* side_multiplier

print(f"Risk per unit: {risk_per_unit:.2f}, Profit per unit: {profit_per_unit:.2f}, Units: {units:.2f}")
# if units < min_qty :
#     units = min_qty
#     print(f"Risk amount too low, setting units to minimum quantity: {units}")

position_value = units * order_price #* margin_leverage
potential_loss = units * risk_per_unit #* margin_leverage
potential_profit = units * profit_per_unit #* margin_leverage


# print(f"Calculated Units: {units:.6f}, Position Value: {position_value:.2f}, Potential Loss: {potential_loss:.2f}, Potential Profit: {potential_profit:.2f}")



Current Price: 2690.7800, Order Price: 2687.0000, Take Profit: 2714.0000, Stop Loss: 2675.0000
Risk per unit: 12.00, Profit per unit: 27.00, Units: 0.33


In [ ]:

# First Margin Order Configuration:  Be VERY careful with amounts!
order_side = "BUY" if trade_type == "long" else "SELL"  # or "SELL" for short
order_type = "MARKET"  # "MARKET", "LIMIT", etc.        

# order_quantity = ensure_multiple_of_step_size(units, step_size)
# step_size = float(step_size)
step_size = 0.00001

precision = int(round(-math.log(step_size, 10), 0))
order_quantity = float(round(units, precision))
# order_quantity = units
# order_quantity = "{:0.0{}f}".format(units, precision)
# print(f"Order Quantity: {order_quantity}, units: {units}, step_size: {step_size}")
print(f"Order Quantity: {order_quantity}, units: {units}")

# OCO Order Configuration (Take Profit and Stop Loss)
oco_side = "SELL" if order_side == "BUY" else "BUY"  # Opposite of the margin order
# oco_quantity = ensure_multiple_of_step_size(units * (1 - fee_percent), step_size)
oco_quantity = order_quantity

# Create the initial margin order
# order_response = client.new_order(
#     symbol=symbol,
#     side=order_side,
#     type=order_type,
#     quantity=order_quantity,
#     # timeInForce='GTC'
#     # price=59808
# )

# if order_response:
#     print("Futures Order Response:", order)
#     aboveType="TAKE_PROFIT" if order_side == "BUY" else "STOP_LOSS"
#     belowType="TAKE_PROFIT" if order_side == "SELL" else "STOP_LOSS"
#     # Create the OCO order immediately after
#     oco_order_response = order = client.create_oco_order(
#         symbol=symbol,
#         side=oco_side,
#         quantity=oco_quantity,
#         aboveType=aboveType,
#         belowType=belowType,
#         abovePrice=take_profit_price if order_side == "BUY" else stop_loss_price,
#         belowPrice=take_profit_price if order_side == "SELL" else stop_loss_price,
#     )

#     if oco_order_response:
#         print("FUTURES OCO Order Response:", oco_order_response)
#     else:
#         print("Failed to create FUTURES OCO order.")
# else:
#     print("Failed to create initial FUTURES order.")


# leverage_response = client.futures_change_leverage(symbol=symbol, leverage=margin_leverage)
# print(leverage_response)
# margin_leverage = leverage_response['leverage']
# print(f'Margin Leverage: changed to {margin_leverage}')

# Example usage (assuming all_pairs, exchange_info, client are defined elsewhere)
# place_futures_order_with_oco(
#     symbol,
#     take_profit_price,
#     stop_loss_price,
#     risk_amount,
#     margin_leverage,
#     trade_type=trade_type,
# )



Current Price: 2717.39, Take Profit: 2774, Stop Loss: 2700
Risk per unit: 17.389999999999873, Profit per unit: 56.61000000000013, Units: 0.23001725129384873
Calculated Units: 0.230017, Position Value: 3125.23, Potential Loss: 20.00, Potential Profit: 65.11
Order Quantity: 0.23002, units: 0.23001725129384873


In [ ]:
# client.basis("BTCUSD", "PERPETUAL", "1d")

[{'indexPrice': '66769.28071912',
  'contractType': 'PERPETUAL',
  'basisRate': '-0.0004',
  'futuresPrice': '66745.8',
  'annualizedBasisRate': '',
  'basis': '-23.48071912',
  'pair': 'BTCUSD',
  'timestamp': 1722297600000},
 {'indexPrice': '66171.94334566',
  'contractType': 'PERPETUAL',
  'basisRate': '-0.0004',
  'futuresPrice': '66143.7',
  'annualizedBasisRate': '',
  'basis': '-28.24334566',
  'pair': 'BTCUSD',
  'timestamp': 1722384000000},
 {'indexPrice': '64619.50957030',
  'contractType': 'PERPETUAL',
  'basisRate': '-0.0006',
  'futuresPrice': '64578.0',
  'annualizedBasisRate': '',
  'basis': '-41.50957030',
  'pair': 'BTCUSD',
  'timestamp': 1722470400000},
 {'indexPrice': '65294.02007067',
  'contractType': 'PERPETUAL',
  'basisRate': '-0.0001',
  'futuresPrice': '65290.0',
  'annualizedBasisRate': '',
  'basis': '-4.02007067',
  'pair': 'BTCUSD',
  'timestamp': 1722556800000},
 {'indexPrice': '61416.47607221',
  'contractType': 'PERPETUAL',
  'basisRate': '-0.0005',
  

In [44]:

symbol = f"{base_asset}{quote_asset}_PERP"
symbol_info = [pair for pair in all_pairs if pair['symbol'] == symbol][0]
# symbol_info['filters']
# and pair['symbol'] == f"{pair['baseAsset']}{pair['quoteAsset']}_PERP"
# symbol
# symbol_info = [s for s in exchange_info['symbols'] if s['symbol'] == symbol][0]

step_size = ([f for f in symbol_info['filters'] if f['filterType'] == 'LOT_SIZE'][0]['stepSize'])
# [f for f in symbol_info['filters']]
step_size
# ticker = client.ticker_price(symbol)
# current_price = float(ticker['price'])
client.ticker_price(symbol)


[{'symbol': 'BTCUSD_PERP',
  'ps': 'BTCUSD',
  'price': '96806.7',
  'time': 1739493089064}]

In [23]:
all_pairs

[{'symbol': 'BTCUSD_PERP',
  'pair': 'BTCUSD',
  'contractType': 'PERPETUAL',
  'deliveryDate': 4133404800000,
  'onboardDate': 1597042800000,
  'contractStatus': 'TRADING',
  'contractSize': 100,
  'marginAsset': 'BTC',
  'maintMarginPercent': '2.5000',
  'requiredMarginPercent': '5.0000',
  'baseAsset': 'BTC',
  'quoteAsset': 'USD',
  'pricePrecision': 1,
  'quantityPrecision': 0,
  'baseAssetPrecision': 8,
  'quotePrecision': 8,
  'equalQtyPrecision': 4,
  'maxMoveOrderLimit': 10000,
  'triggerProtect': '0.0500',
  'underlyingType': 'COIN',
  'underlyingSubType': ['PoW'],
  'filters': [{'minPrice': '1000',
    'maxPrice': '4520958',
    'filterType': 'PRICE_FILTER',
    'tickSize': '0.1'},
   {'stepSize': '1',
    'filterType': 'LOT_SIZE',
    'maxQty': '1000000',
    'minQty': '1'},
   {'stepSize': '1',
    'filterType': 'MARKET_LOT_SIZE',
    'maxQty': '60000',
    'minQty': '1'},
   {'limit': 200, 'filterType': 'MAX_NUM_ORDERS'},
   {'limit': 20, 'filterType': 'MAX_NUM_ALGO_ORDER

In [49]:
import requests
from bs4 import BeautifulSoup
import markdownify
import os

def record_webpage_to_markdown(url, output_filename="output.md"):
    """
    Records the content of a webpage at the given URL into a Markdown file.

    Args:
        url (str): The URL of the webpage to record.
        output_filename (str): The name of the output Markdown file (default: "output.md").
    """
    try:
        # 1. Fetch the webpage
        response = requests.get(url)
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        html = response.text

        # 2. Parse the HTML
        soup = BeautifulSoup(html, 'html.parser')

        # Remove script and style tags (optional, but often desirable)
        for script in soup(["script", "style"]):
            script.extract()

        # 3. Convert HTML to Markdown
        markdown = markdownify.markdownify(str(soup))

        # 4. Save to a file
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(markdown)

        print(f"Webpage content recorded to {output_filename}")

    except requests.exceptions.RequestException as e:
        print(f"Error fetching webpage: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    webpage_url = "https://binance-connector.readthedocs.io/en/latest/binance.spot.margin.html#get-all-margin-assets-market-data"  # Replace with the URL you want to record
    output_file = "binance-margin.md"             # Replace with your desired output filename

    record_webpage_to_markdown(webpage_url, output_file)

2025-02-12 22:29:29.555 UTC DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): binance-connector.readthedocs.io:443
2025-02-12 22:29:29.713 UTC DEBUG urllib3.connectionpool: https://binance-connector.readthedocs.io:443 "GET /en/latest/binance.spot.margin.html HTTP/1.1" 200 None


Webpage content recorded to binance-margin.md


In [68]:
step_size

0.001

In [ ]:
## Start: 22-04-01
## End: 22-06-01
## Timeframe: 4h
## Indicators: Fibonacci retracements off highs and lows
"""
Loads of losing long positions in a downtrend. Should pay more attention to ranges.
I don't pick up trend reversals early enough.
"""
t = np.array([ 
    0, -1, 2.02, 2.23, 3.4, -1, 3.78, -1, -1, -1, -1, 2, -1, -1, -1, -1, 3.34, -1, -1, -1, -1, -1, -1, 2.08,
    -1, -1, -1, -1, 2.6, 3.17, 2.32, -1, -1, -1, 2.61, 2.14, -1, -1, -1, -1, -1, -1, 2.21, -1, -1, -1, -1,
    2.27, -1, -1, -1, -1, -1, -1, 2.71, -1, 2.10, 2.42, -1, -1, -1, 2.17, -1, -1, 2.39, -1, -1, 2.75, -1, -1, -1,
    3.03, -1, -1, 2.4, -1, -1, 2.75, -1, -1, -1, -1, -1, -1, -1, 2.03, -1, 2.74, -1, -1
    
])
pnl = np.sum(t)
ntrades = len(t)
win_ratio = np.sum(t > 0) / len(t)
print(f'pnl: {pnl:.2f},  ntrades: {ntrades}, win-ratio:{win_ratio:.2f}')


In [ ]:
## Start: 22-04-01
## End: 22-06-01
## Timeframe: 1h
## Indicators: Fibonacci retracements off highs and lows
"""
Loads of losing long positions in a downtrend. Should pay more attention to ranges.
I don't pick up trend reversals early enough.
"""
t = np.array([ 
    -1, 2.62, -1, 2.22, -1, -1, -1, 2.25, 2.18, -1, 2.94, -1, -1, 2.33, -1, 3.15, -1, -1, -1, -1, 2.98, 3.49, 3.49, -1, -1, -1, -1, -1, 3.27, -1, -1, -1, -1, 3.7, -1, -1, -1, 3.17, -1, 2.03, -1, 5.16, -1, -1
    
])
pnl = np.sum(t)
ntrades = len(t)
win_ratio = np.sum(t > 0) / len(t)
print(f'pnl: {pnl:.2f},  ntrades: {ntrades}, win-ratio:{win_ratio:.2f}')
